# ICTS-ICTP Winter School on Quantitative Systems Biology 2025

This notebook demonstrates how to recover dynamical equations from time series data, using a stochastic version of a two-species Lotka-Volterra model as an example.
We use the [PyDaddy](http://github.com/tee-lab/PyDaddy) Python package for SDE discovery. 

This is a tutorial notebook prepared as part of the [ICTS-ICTP Winter School on Quantitative Systems Biology 2025](https://www.icts.res.in/program/qsb2024).

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('/Users/nabeel/Documents/4 Archive/[2024.11] Archive/Research/Code/pydaddy')

import pydaddy

# Simulating a stochastic Lotka-Volterra model

This section contains code to simulate a stochastic Lotka-Volterra competition model. The model can be described using the following differential equations:
$$
\begin{align}
\frac{dx}{dt} = 2x \left( 1 - \frac{1}{6}(x + 2y) \right) + 0.2 x \cdot \eta_x(t) \\
\frac{dy}{dt} = 4y \left( 1 - \frac{1}{8}(2x + y) \right) + 0.2 y \cdot \eta_y(t) \\
\end{align}
$$

This form (i.e. linear in population sizes) of noise is usually called _environmental noise_ in the literature.
The details of the simulation are not important for this tutorial, so feel free to skip ahead to the next section.


In [3]:
rng = np.random.default_rng(seed=42)

def f(z):
    x, y = z[0], z[1]
    dx = 2 * x * (1 - (x + 2 * y) / 6)
    dy = 4 * y * (1 - (2 * x + y) / 8)
    return np.array([dx, dy])


def g(z):
    x, y = z[0], z[1]
    return np.array([0.2 * x, 0.2 * y])


# def f(z):
#     x, y = z[0], z[1]
#     return np.array((-2*x, -3*y))

# def g(z):
#     return np.array((0, 0))


def simulate_lv(T=10000, dt=0.01, x0=(1, 1)):
    xs = np.full((T, 2), np.nan)
    xs[0] = x0
    # dW = rng.standard_normal((T, 2)) * np.sqrt(dt)
    dW = rng.normal(0, 1, (T, 2)) * np.sqrt(dt)
    for t in range(T-1):
        xs[t+1] = xs[t] + f(xs[t]) * dt + g(xs[t]) * dW[t]

    return xs

In [23]:
# Simulate the model, same parameters, for different initial conditions.
N = 30  # Number of trials
T = 5000  # Number of timesteps in each trial
dt = 0.001

x0s = np.linspace((1, 10), (10, 1), N)
# y0s = np.flipud(x0s)

xs = np.full((N, T + 1, 2), np.nan)
for i in range(N):
    xs[i, :-1] = simulate_lv(T, dt=dt, x0=x0s[i])
# xs = simulate_lv(5000, dt=0.0001, x0=(5, 5))
# xs.shape

In [ ]:
plt.plot(xs[15])
plt.show()

In [35]:
xs = np.reshape(xs, (-1, 2), order='C')

# Estimating dynamical equations from time series

The above section simulated a time series using a model. Now suppose that this is the trajectory data of the population abundances of two populations, with interactions unknown to us. We will use the data-driven SDE discovery framework (with the PyDaddy package) to estimate a model to describe the dynamics of these trajectories.

First, we initialize a PyDaddy object using `pydaddy.Characterize`, passing in as arguments the time series, and the sampling interval $\Delta t$.

In [36]:
dd = pydaddy.Characterize(xs.T, t=dt, show_summary=False)

Now, we call the appropriate `fit()` methods to fit functional forms for the drift and diffusion functions. We do this individually for each component of the drift and diffusion functions.

While fitting, one need to specify the maximum order of the polynomial (thereby fixing the library), and the sparsification `threshold` parameter. One may often need to play around these to find the right choices. For more information on this, see [PyDaddy tutorials](https://pydaddy.readthedocs.io/en/latest/tutorials.html).


In [ ]:
dd.fit('F1', order=3, threshold=0.1)

In [ ]:
dd.fit('F2', order=3, threshold=0.2)

In [ ]:
dd.fit('G11', order=2, threshold=0.03)

In [ ]:
dd.fit('G22', order=2, threshold=0.03)

Compare the expressions for the drift and diffusion with the original model that we started with. Were we able to recover the original model accurately?

We can visualize the fitted functions using `drift()` and `diffusion()` respectively.

In [ ]:
dd.drift()

In [ ]:
dd.diffusion()

# Suggested Exercises
0. Go to the [PyDaddy tutorials page](https://pydaddy.readthedocs.io/en/latest/tutorials.html) and try out the first two tutorials to get a feel for the package.
1. Play around with the `order` and `threshold` parameters of the `fit()` function above. When does the fitting performance degrade?
2. Use only a subset of the full time series `xs` (for example, only the first 10000 or 1000 time points). What happens to the drift and diffusion estimates when you have very little data?
3. Subsample the data with larger $\Delta t$ - you can do this by using `x_sub = xs[::factor] whare `factor` is an (integer) subsampling factor. Use the subsampled time series to do the fitting. Are we able to recover the original model accurately?
2. (Advanced) Modify the simulation code to simulate other models (different parameters of LV, models with Allee effects, etc.) and experiment with the model discovery procedure. 